# BERTopic: Topic Evolution Quality (TTC, TTS, TTQ)

**Scenario 2** — Evaluates whether topic transitions are smooth and coherent.
- **TTC**: Temporal Topic Coherence — Normalized NPMI of strict cross-time word pairs (t × t+1) against full corpus
- **TTS**: Temporal Topic Stability — RBO similarity of word rankings between t and t+1
- **TTQ**: Harmonic mean of TTC and TTS

In [1]:
import ast
import time
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [2]:
MODEL_NAME = "bertopic"
LIST_SUBJECT = ["cs", "math", "physics"]
DATA_DIR = Path("../../../../data/preprocess")
TEMPORAL_DIR = Path("../../../../results/bertopic/temporal")
RESULT_DIR = Path("../../../../results/bertopic/evolution")
DATA_FORMAT = "emb"
VERSION = "v1"
TOP_N = 10

for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

In [3]:
def parse_text(x):
    return str(x).split()


def build_word_doc_sets(corpus_texts):
    """Build a dict mapping each word to the set of doc indices containing it."""
    word_docs = {}
    for i, tokens in enumerate(corpus_texts):
        for w in set(tokens):
            if w not in word_docs:
                word_docs[w] = set()
            word_docs[w].add(i)
    return word_docs


def compute_ttc_npmi(words_t, words_t1, word_docs, n_docs):
    """Compute TTC as normalized average NPMI of strict cross-time word pairs.
    
    Pairs: (w_i from t) x (w_j from t+1), checked against full corpus.
    Raw NPMI ranges [-1, 1], normalized to [0, 1] via (npmi + 1) / 2.
    """
    if not words_t or not words_t1 or n_docs == 0:
        return 0.0
    eps = 1e-12
    npmi_scores = []
    for w_i in words_t:
        for w_j in words_t1:
            docs_i = word_docs.get(w_i, set())
            docs_j = word_docs.get(w_j, set())
            if len(docs_i) == 0 or len(docs_j) == 0:
                continue
            if w_i == w_j:
                npmi_scores.append(1.0)
                continue
            co_occur = len(docs_i & docs_j)
            if co_occur == 0:
                npmi_scores.append(-1.0)
                continue
            p_ij = co_occur / n_docs
            p_i = len(docs_i) / n_docs
            p_j = len(docs_j) / n_docs
            pmi = np.log((p_ij + eps) / (p_i * p_j + eps))
            npmi = pmi / (-np.log(p_ij + eps))
            npmi_scores.append(npmi)
    if not npmi_scores:
        return 0.0
    raw = float(np.mean(npmi_scores))
    # Normalize from [-1, 1] to [0, 1]
    return (raw + 1) / 2


def compute_rbo(list1, list2, p=0.9):
    """Rank-Biased Overlap between two ranked word lists."""
    if not list1 or not list2:
        return 0.0
    k = min(len(list1), len(list2))
    score = 0.0
    for d in range(1, k + 1):
        s1 = set(list1[:d])
        s2 = set(list2[:d])
        overlap = len(s1 & s2) / d
        score += (p ** (d - 1)) * overlap
    return 1 - (score * (1 - p))


def harmonic_mean(a, b):
    """F1-style harmonic mean."""
    if (a + b) <= 0:
        return 0.0
    return 2 * a * b / (a + b)

## Compute TTC, TTS, TTQ per transition

In [4]:
for subject in LIST_SUBJECT:
    print(f"\n{'='*70}")
    print(f"Topic Evolution: {subject.upper()} (BERTopic)")
    print(f"{'='*70}")

    # Load topic word evolution
    evo_df = pd.read_csv(TEMPORAL_DIR / subject / "topic_word_evolution.csv")

    # Load FULL corpus as reference for NPMI
    data_df = pd.read_csv(DATA_DIR / subject / DATA_FORMAT / f"{VERSION}.csv")
    data_df["year"] = pd.to_datetime(data_df["submitted_date"]).dt.year
    corpus_texts = [parse_text(x) for x in data_df["text"]]
    n_docs = len(corpus_texts)

    # Build word-doc co-occurrence index over FULL corpus
    print(f"  Building word-doc index over {n_docs:,} documents...")
    word_docs = build_word_doc_sets(corpus_texts)
    print(f"  Vocabulary size: {len(word_docs):,}")

    # Parse topic words per (year, topic_id)
    topic_words = {}
    for _, row in evo_df.iterrows():
        key = (int(row["year"]), int(row["topic_id"]))
        words = [w.strip() for w in str(row["top_words"]).split(",")][:TOP_N]
        topic_words[key] = words

    years = sorted(data_df["year"].unique())
    all_topics = sorted(evo_df["topic_id"].unique())

    all_rows = []
    transition_rows = []

    for i in range(len(years) - 1):
        t, t_next = int(years[i]), int(years[i + 1])
        start = time.time()

        # Find topics valid at both t and t+1
        valid_tids = [tid for tid in all_topics
                      if (t, tid) in topic_words and (t_next, tid) in topic_words
                      and len(topic_words[(t, tid)]) >= 2]

        # Per-topic metrics
        ttc_list, tts_list, ttq_list = [], [], []
        for tid in valid_tids:
            w_t = topic_words[(t, tid)]
            w_t1 = topic_words[(t_next, tid)]

            # TTC: Normalized NPMI of strict cross-time pairs against full corpus
            ttc = compute_ttc_npmi(w_t, w_t1, word_docs, n_docs)

            # TTS: RBO similarity of ranked word lists
            tts = compute_rbo(w_t, w_t1, p=0.9)

            ttq = harmonic_mean(ttc, tts)

            ttc_list.append(ttc)
            tts_list.append(tts)
            ttq_list.append(ttq)

            all_rows.append({"subject": subject, "year_from": t, "year_to": t_next,
                             "topic_id": int(tid),
                             "ttc": round(ttc, 6), "tts": round(tts, 6), "ttq": round(ttq, 6)})

        avg_ttc = np.mean(ttc_list) if ttc_list else 0.0
        avg_tts = np.mean(tts_list) if tts_list else 0.0
        avg_ttq = np.mean(ttq_list) if ttq_list else 0.0
        elapsed = time.time() - start

        transition_rows.append({"subject": subject, "year_from": t, "year_to": t_next,
                                "n_topics": len(valid_tids),
                                "avg_ttc": round(avg_ttc, 6),
                                "avg_tts": round(avg_tts, 6),
                                "avg_ttq": round(avg_ttq, 6)})

        print(f"  {t}→{t_next}: TTC={avg_ttc:.4f}  TTS={avg_tts:.4f}  "
              f"TTQ={avg_ttq:.4f}  ({len(valid_tids)} topics) [{elapsed:.1f}s]")

    # Save per-topic metrics
    pd.DataFrame(all_rows).to_csv(
        RESULT_DIR / subject / "topic_evolution_metrics.csv", index=False)

    # Save transition summary
    trans_df = pd.DataFrame(transition_rows)
    trans_df.to_csv(RESULT_DIR / subject / "transition_summary.csv", index=False)

    # Overall summary
    subj_trans = trans_df[trans_df["subject"] == subject]
    overall = {"subject": subject,
               "avg_ttc": round(subj_trans["avg_ttc"].mean(), 6),
               "std_ttc": round(subj_trans["avg_ttc"].std(), 6),
               "avg_tts": round(subj_trans["avg_tts"].mean(), 6),
               "std_tts": round(subj_trans["avg_tts"].std(), 6),
               "avg_ttq": round(subj_trans["avg_ttq"].mean(), 6),
               "std_ttq": round(subj_trans["avg_ttq"].std(), 6)}
    pd.DataFrame([overall]).to_csv(
        RESULT_DIR / subject / "evolution_summary.csv", index=False)

    print(f"\n  Overall: TTC={overall['avg_ttc']:.4f}±{overall['std_ttc']:.4f}  "
          f"TTS={overall['avg_tts']:.4f}±{overall['std_tts']:.4f}  "
          f"TTQ={overall['avg_ttq']:.4f}±{overall['std_ttq']:.4f}")
    print(f"  Saved to: {RESULT_DIR / subject}")


Topic Evolution: CS (BERTopic)
  Building word-doc index over 165,756 documents...
  Vocabulary size: 166,520
  2000→2001: TTC=0.3466  TTS=0.9126  TTQ=0.4817  (36 topics) [0.0s]
  2001→2002: TTC=0.3829  TTS=0.9026  TTQ=0.5189  (39 topics) [0.1s]
  2002→2003: TTC=0.3701  TTS=0.9087  TTQ=0.5037  (40 topics) [0.1s]
  2003→2004: TTC=0.3459  TTS=0.9187  TTQ=0.4857  (50 topics) [0.1s]
  2004→2005: TTC=0.3768  TTS=0.9166  TTQ=0.5192  (56 topics) [0.1s]
  2005→2006: TTC=0.3815  TTS=0.9007  TTQ=0.5107  (54 topics) [0.1s]
  2006→2007: TTC=0.3825  TTS=0.9084  TTQ=0.5181  (59 topics) [0.1s]
  2007→2008: TTC=0.3823  TTS=0.8999  TTQ=0.5146  (62 topics) [0.1s]
  2008→2009: TTC=0.3815  TTS=0.9083  TTQ=0.5142  (72 topics) [0.1s]
  2009→2010: TTC=0.3980  TTS=0.8886  TTQ=0.5314  (79 topics) [0.1s]
  2010→2011: TTC=0.3875  TTS=0.8969  TTQ=0.5210  (97 topics) [0.2s]
  2011→2012: TTC=0.3880  TTS=0.8792  TTQ=0.5158  (125 topics) [0.2s]
  2012→2013: TTC=0.4037  TTS=0.8709  TTQ=0.5305  (147 topics) [0.2s]
  2

## Evolution Summary

In [5]:
print(f"\n{'='*70}")
print(f"BERTopic — Evolution Summary")
print(f"{'='*70}")

for subject in LIST_SUBJECT:
    summary = pd.read_csv(RESULT_DIR / subject / "evolution_summary.csv").iloc[0]
    trans = pd.read_csv(RESULT_DIR / subject / "transition_summary.csv")

    print(f"\n  {subject.upper()}:")
    print(f"    TTC = {summary['avg_ttc']:.4f} ± {summary['std_ttc']:.4f}")
    print(f"    TTS = {summary['avg_tts']:.4f} ± {summary['std_tts']:.4f}")
    print(f"    TTQ = {summary['avg_ttq']:.4f} ± {summary['std_ttq']:.4f}")
    print(f"    TTQ range: {trans['avg_ttq'].min():.4f} — {trans['avg_ttq'].max():.4f}")


BERTopic — Evolution Summary

  CS:
    TTC = 0.4327 ± 0.0720
    TTS = 0.8219 ± 0.1057
    TTQ = 0.5345 ± 0.0278
    TTQ range: 0.4817 — 0.5731

  MATH:
    TTC = 0.4794 ± 0.0611
    TTS = 0.7904 ± 0.0846
    TTQ = 0.5716 ± 0.0247
    TTQ range: 0.5271 — 0.6016

  PHYSICS:
    TTC = 0.4526 ± 0.0542
    TTS = 0.8254 ± 0.0717
    TTQ = 0.5555 ± 0.0246
    TTQ range: 0.5172 — 0.5866
